# Project 1 — Clinical Trial RAG: Data Pipeline

Day 7 build. Goal: pull trials → chunk → embed → FAISS → retrieve → Groq generate.

In [1]:
import sentence_transformers, faiss, groq, requests
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
import os

load_dotenv("../.env")  # .env lives one level up from notebooks/
key = os.getenv("GROQ_API_KEY")

print(f"sentence-transformers: {sentence_transformers.__version__}")
print(f"faiss:                 {faiss.__version__}")
print(f"groq:                  {groq.__version__}")
print(f"langchain splitters:   loaded")
print(f"GROQ_API_KEY loaded:   {bool(key)}  (length={len(key) if key else 0})")

sentence-transformers: 5.4.1
faiss:                 1.13.2
groq:                  1.2.0
langchain splitters:   loaded
GROQ_API_KEY loaded:   True  (length=56)


In [ ]:
#Cell 2 — One API call, look at the shape
import requests, json

BASE_URL = "https://clinicaltrials.gov/api/v2/studies"

# One trial about pembrolizumab (the same drug that broke MiniLM in Day 6)
params = {
    "query.term": "pembrolizumab lung cancer",
    "pageSize": 1,
    "format": "json"
}

response = requests.get(BASE_URL, params=params, timeout=30)
print(f"Status: {response.status_code}")
print(f"URL:    {response.url}\n")

data = response.json()
print(f"Top-level keys: {list(data.keys())}")
print(f"Number of studies returned: {len(data.get('studies', []))}\n")

# Pretty-print the first trial so you can see what fields exist
first_trial = data["studies"][0]
print(json.dumps(first_trial, indent=2)[:2000])  # first 2000 chars only
print("\n... (truncated)")

Status: 200
URL:    https://clinicaltrials.gov/api/v2/studies?query.term=pembrolizumab+lung+cancer&pageSize=1&format=json

Top-level keys: ['studies', 'nextPageToken']
Number of studies returned: 1

{
  "protocolSection": {
    "identificationModule": {
      "nctId": "NCT07094113",
      "orgStudyIdInfo": {
        "id": "20240031"
      },
      "organization": {
        "fullName": "Amgen",
        "class": "INDUSTRY"
      },
      "briefTitle": "AMG 410 Alone and in Combination With Other Agents in Participants With KRAS Altered Advanced or Metastatic Solid Tumors",
      "officialTitle": "A Phase 1/1b Study Evaluating the Safety, Tolerability, Pharmacokinetics, Pharmacodynamics, and Efficacy of AMG 410 Alone and in Combination With Other Agents in Participants With KRAS Altered Advanced or Metastatic Solid Tumors"
    },
    "statusModule": {
      "statusVerifiedDate": "2026-02",
      "overallStatus": "RECRUITING",
      "expandedAccessInfo": {
        "hasExpandedAccess": fals

In [ ]:
#Cell 3: Build a paginated puller
def fetch_trials(query_term, n_trials=100, page_size=100):
    """
    Pull N clinical trials matching query_term from ClinicalTrials.gov v2 API.
    Paginates automatically using nextPageToken.
    """
    all_studies = []
    next_page_token = None
    page_num = 0

    while len(all_studies) < n_trials:
        page_num += 1
        params = {
            "query.term": query_term,
            "pageSize": min(page_size, n_trials - len(all_studies)),
            "format": "json"
        }
        if next_page_token:
            params["pageToken"] = next_page_token

        response = requests.get(BASE_URL, params=params, timeout=30)
        if response.status_code != 200:
            print(f"  API returned {response.status_code} on page {page_num} — stopping")
            break

        data = response.json()
        studies = data.get("studies", [])
        if not studies:
            print(f"  No more results for '{query_term}' after {len(all_studies)} trials")
            break

        all_studies.extend(studies)
        next_page_token = data.get("nextPageToken")
        if not next_page_token:
            break

    return all_studies[:n_trials]


# Test the puller with a small sample first — 50 trials
sample = fetch_trials("lung cancer immunotherapy", n_trials=50)
print(f"Pulled {len(sample)} trials\n")

print("First 5 NCT IDs and titles:")
for trial in sample[:5]:
    ident = trial["protocolSection"]["identificationModule"]
    nct = ident["nctId"]
    title = ident["briefTitle"]
    print(f"  {nct}: {title[:80]}")

Pulled 50 trials

First 5 NCT IDs and titles:
  NCT05486988: ctDNA as a Biomarker for Treatment in Advanced NSCLC
  NCT01599559: Randomized, Open-label, Two-arms, Phase III Comparative Study Assessing the Role
  NCT03125603: Side Effects of Anti-PD-(L)-1 and Anti CTLA-A4 in the Non Small Cells Lung Cance
  NCT07554846: Comparison of Perioperative Immunotherapy, Adjuvant Immunotherapy or Neoadjuvant
  NCT04192682: Anlotinib Combined With Sintilimab as Second-line Treatment or Beyond in Patient


In [ ]:
#Cell 4: Pull the full 500-trial corpus
import time

# 5 disease areas, 100 trials each = 500 total
QUERIES = [
    "lung cancer immunotherapy",
    "breast cancer chemotherapy",
    "melanoma clinical trial",
    "diabetes type 2 treatment",
    "heart failure cardiovascular",
]

all_trials = []
for q in QUERIES:
    print(f"Fetching: {q!r}...")
    trials = fetch_trials(q, n_trials=100)
    print(f"  -> got {len(trials)} trials")
    all_trials.extend(trials)
    time.sleep(0.5)  # be polite to the API

print(f"\nTotal trials pulled: {len(all_trials)}")

# Deduplicate by NCT ID (a trial about 'lung cancer immunotherapy' might also
# match 'melanoma' if it's a basket study covering multiple cancers)
seen_ids = set()
unique_trials = []
for trial in all_trials:
    nct_id = trial["protocolSection"]["identificationModule"]["nctId"]
    if nct_id not in seen_ids:
        seen_ids.add(nct_id)
        unique_trials.append(trial)

print(f"After dedup:         {len(unique_trials)}")
print(f"Duplicates removed:  {len(all_trials) - len(unique_trials)}")

Fetching: 'lung cancer immunotherapy'...
  -> got 100 trials
Fetching: 'breast cancer chemotherapy'...
  -> got 100 trials
Fetching: 'melanoma clinical trial'...
  -> got 100 trials
Fetching: 'diabetes type 2 treatment'...
  -> got 100 trials
Fetching: 'heart failure cardiovascular'...
  -> got 100 trials

Total trials pulled: 500
After dedup:         484
Duplicates removed:  16


In [ ]:
#Cell 5: Flatten trials to text documents
import pickle


def flatten_trial(trial):
    """
    Extract relevant fields from a trial JSON and join into a single text blob.
    Uses safe .get() with defaults so missing fields don't break the pipeline.
    """
    protocol = trial.get("protocolSection", {})

    ident = protocol.get("identificationModule", {})
    nct_id = ident.get("nctId", "UNKNOWN")
    title = ident.get("briefTitle", "")

    desc = protocol.get("descriptionModule", {})
    brief_summary = desc.get("briefSummary", "")
    detailed_desc = desc.get("detailedDescription", "")

    cond_module = protocol.get("conditionsModule", {})
    conditions = cond_module.get("conditions", [])
    conditions_text = ", ".join(conditions) if conditions else ""

    elig = protocol.get("eligibilityModule", {})
    elig_criteria = elig.get("eligibilityCriteria", "")

    parts = [f"Title: {title}"]
    if conditions_text:
        parts.append(f"Conditions: {conditions_text}")
    if brief_summary:
        parts.append(f"Brief Summary: {brief_summary}")
    if detailed_desc:
        parts.append(f"Detailed Description: {detailed_desc}")
    if elig_criteria:
        parts.append(f"Eligibility: {elig_criteria}")

    text = "\n\n".join(parts)

    return {
        "nct_id": nct_id,
        "title": title,
        "text": text,
        "conditions": conditions,
    }


documents = [flatten_trial(t) for t in unique_trials]

MIN_LENGTH = 200
filtered = [d for d in documents if len(d["text"]) >= MIN_LENGTH]
print(f"Documents flattened:    {len(documents)}")
print(f"Dropped (too short):    {len(documents) - len(filtered)}")
print(f"Final document count:   {len(filtered)}")

documents = filtered

text_lengths = [len(d["text"]) for d in documents]
text_lengths_sorted = sorted(text_lengths)
print(f"\nText length stats:")
print(f"  Min:    {min(text_lengths):,} chars")
print(f"  Max:    {max(text_lengths):,} chars")
print(f"  Mean:   {sum(text_lengths) / len(text_lengths):,.0f} chars")
print(f"  Median: {text_lengths_sorted[len(text_lengths) // 2]:,} chars")

output_path = "../data/trials.pkl"
with open(output_path, "wb") as f:
    pickle.dump(documents, f)
print(f"\nSaved {len(documents)} documents to {output_path}")

print("\n" + "=" * 70)
print("SAMPLE DOCUMENT")
print("=" * 70)
sample = documents[0]
print(f"NCT ID:      {sample['nct_id']}")
print(f"Title:       {sample['title']}")
print(f"Conditions:  {sample['conditions']}")
print(f"Text length: {len(sample['text']):,} chars")
print(f"\nFirst 800 chars of text:")
print(sample["text"][:800])
print("...")

Documents flattened:    484
Dropped (too short):    0
Final document count:   484

Text length stats:
  Min:    360 chars
  Max:    26,043 chars
  Mean:   4,616 chars
  Median: 3,482 chars

Saved 484 documents to ../data/trials.pkl

SAMPLE DOCUMENT
NCT ID:      NCT05486988
Title:       ctDNA as a Biomarker for Treatment in Advanced NSCLC
Conditions:  ['Non-small Cell Lung Cancer']
Text length: 2,417 chars

First 800 chars of text:
Title: ctDNA as a Biomarker for Treatment in Advanced NSCLC

Conditions: Non-small Cell Lung Cancer

Brief Summary: The dynamic monitoring of circulating tumor DNA aims to evaluate the response and progression-free survival of short-course chemotherapy (2 cycles) combined with immunotherapy in patients with locally advanced unresectable or metastatic non-small cell lung cancer.

Detailed Description: For patients with locally advanced unresectable or metastatic non-small cell lung cancers, 4-6 cycles of chemotherapy plus immunotherapy with immune maintenance 

In [ ]:
# Cell 6: Recursive chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len,
)

# Chunk every document, preserving NCT ID and title as metadata so
# we can cite the source trial when we display the answer.
chunks = []
for doc in documents:
    text_chunks = splitter.split_text(doc["text"])
    for i, chunk_text in enumerate(text_chunks):
        chunks.append({
            "chunk_id": f"{doc['nct_id']}_chunk_{i}",
            "nct_id": doc["nct_id"],
            "title": doc["title"],
            "text": chunk_text,
        })

print(f"Total documents:  {len(documents)}")
print(f"Total chunks:     {len(chunks):,}")
print(f"Avg chunks/trial: {len(chunks) / len(documents):.1f}")

chunk_lengths = [len(c["text"]) for c in chunks]
print(f"\nChunk length stats:")
print(f"  Min:    {min(chunk_lengths)} chars")
print(f"  Max:    {max(chunk_lengths)} chars")
print(f"  Mean:   {sum(chunk_lengths) / len(chunk_lengths):.0f} chars")

with open("../data/chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)
print(f"\nSaved {len(chunks):,} chunks to ../data/chunks.pkl")

# Inspect chunks from the same trial we looked at earlier — see the overlap
print("\n" + "=" * 70)
print("CHUNKS FROM NCT05486988 (the ctDNA NSCLC trial)")
print("=" * 70)
sample_chunks = [c for c in chunks if c["nct_id"] == "NCT05486988"]
print(f"This 2,417-char trial split into {len(sample_chunks)} chunks\n")

for i, c in enumerate(sample_chunks):
    print(f"--- Chunk {i} ({len(c['text'])} chars) ---")
    print(c["text"][:300] + ("..." if len(c["text"]) > 300 else ""))
    print()

Total documents:  484
Total chunks:     3,351
Avg chunks/trial: 6.9

Chunk length stats:
  Min:    18 chars
  Max:    1000 chars
  Mean:   701 chars

Saved 3,351 chunks to ../data/chunks.pkl

CHUNKS FROM NCT05486988 (the ctDNA NSCLC trial)
This 2,417-char trial split into 4 chunks

--- Chunk 0 (380 chars) ---
Title: ctDNA as a Biomarker for Treatment in Advanced NSCLC

Conditions: Non-small Cell Lung Cancer

Brief Summary: The dynamic monitoring of circulating tumor DNA aims to evaluate the response and progression-free survival of short-course chemotherapy (2 cycles) combined with immunotherapy in patie...

--- Chunk 1 (789 chars) ---
Detailed Description: For patients with locally advanced unresectable or metastatic non-small cell lung cancers, 4-6 cycles of chemotherapy plus immunotherapy with immune maintenance therapy is currently the standard treatment. Short-course chemotherapy (2 cycles) combined with immunotherapy has bee...

--- Chunk 2 (870 chars) ---
Eligibility: Inclusion 

In [ ]:
#Cell 7: Load PubMedBERT
from sentence_transformers import SentenceTransformer
import time

MODEL_NAME = "pritamdeka/S-PubMedBert-MS-MARCO"

print(f"Loading {MODEL_NAME}...")
print("First time: ~440MB download, 2-5 min depending on connection.")
print("After that: loads from local cache in ~10 seconds.\n")

t0 = time.time()
model = SentenceTransformer(MODEL_NAME)
load_time = time.time() - t0

print(f"Loaded in {load_time:.1f}s")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")
print(f"Max sequence length: {model.max_seq_length} tokens")

# Sanity: encode one chunk and inspect the output shape
sample_text = chunks[0]["text"]
print(f"\nTest encode on chunks[0] ({len(sample_text)} chars):")
embedding = model.encode(sample_text)
print(f"  Output shape:    {embedding.shape}")
print(f"  Dtype:           {embedding.dtype}")
print(f"  First 5 values:  {embedding[:5]}")
print(f"  L2 norm:         {(embedding ** 2).sum() ** 0.5:.4f}")

Loading pritamdeka/S-PubMedBert-MS-MARCO...
First time: ~440MB download, 2-5 min depending on connection.
After that: loads from local cache in ~10 seconds.



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded in 2.6s
Embedding dimension: 768
Max sequence length: 350 tokens

Test encode on chunks[0] (380 chars):


C:\Users\shrik\AppData\Local\Temp\ipykernel_21856\3712572202.py:15: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")


  Output shape:    (768,)
  Dtype:           float32
  First 5 values:  [-0.41392452 -0.46288306 -0.6584433  -0.7457306  -0.23541652]
  L2 norm:         15.1295


In [10]:
#Cell 8: Encode all chunks and build FAISS index
import numpy as np
import faiss

# Filter the tiny tail chunks we flagged earlier (the 18-char outliers)
MIN_CHUNK_LEN = 100
chunks_filtered = [c for c in chunks if len(c["text"]) >= MIN_CHUNK_LEN]
print(f"Chunks before filter:  {len(chunks):,}")
print(f"Chunks after filter:   {len(chunks_filtered):,}")
print(f"Dropped tiny chunks:   {len(chunks) - len(chunks_filtered):,}")
chunks = chunks_filtered

# Encode all chunks. normalize_embeddings=True does the L2-normalization
# in one shot — cleaner than dividing by norm manually after the fact.
texts = [c["text"] for c in chunks]
print(f"\nEncoding {len(texts):,} chunks on CPU...")
t0 = time.time()
embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
encode_time = time.time() - t0
print(f"\nEncoding done in {encode_time:.1f}s  ({len(texts) / encode_time:.1f} chunks/sec)")
print(f"Embeddings shape:     {embeddings.shape}")
print(f"Embeddings dtype:     {embeddings.dtype}")
print(f"Sample L2 norm:       {(embeddings[0] ** 2).sum() ** 0.5:.4f}   (should be ~1.0 now)")

# Build FAISS IndexFlatL2: exhaustive search, perfect recall.
# At 3K vectors this is instant. At 3M you'd switch to IndexIVFFlat or HNSW.
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings.astype(np.float32))
print(f"\nFAISS index built:    {index.ntotal:,} vectors, {dim} dims")

# Persist everything so we don't re-encode on kernel restart
np.save("../data/embeddings.npy", embeddings)
faiss.write_index(index, "../data/faiss.index")
with open("../data/chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

print(f"\nSaved to disk:")
print(f"  ../data/embeddings.npy   ({embeddings.nbytes / 1024 / 1024:.1f} MB)")
print(f"  ../data/faiss.index      ({len(chunks):,} vectors)")
print(f"  ../data/chunks.pkl       ({len(chunks):,} chunks with metadata)")

Chunks before filter:  3,351
Chunks after filter:   3,264
Dropped tiny chunks:   87

Encoding 3,264 chunks on CPU...


Batches:   0%|          | 0/102 [00:00<?, ?it/s]


Encoding done in 396.4s  (8.2 chunks/sec)
Embeddings shape:     (3264, 768)
Embeddings dtype:     float32
Sample L2 norm:       1.0000   (should be ~1.0 now)

FAISS index built:    3,264 vectors, 768 dims

Saved to disk:
  ../data/embeddings.npy   (9.6 MB)
  ../data/faiss.index      (3,264 vectors)
  ../data/chunks.pkl       (3,264 chunks with metadata)


In [11]:
#Cell 9: The retrieval function
def retrieve(query: str, k: int = 5):
    """
    Encode the query, search FAISS for top-k nearest chunks, return them
    with similarity scores.

    Returns: list of dicts with chunk metadata + similarity score.
    Score is cosine similarity in [-1, 1]; higher is more similar.
    """
    query_vec = model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)

    # FAISS L2 distance on unit vectors: distance^2 = 2 - 2*cosine
    # So cosine = 1 - (distance^2 / 2)
    distances, indices = index.search(query_vec, k)

    results = []
    for rank, (dist, idx) in enumerate(zip(distances[0], indices[0])):
        chunk = chunks[idx]
        cosine_sim = 1 - (dist / 2)
        results.append({
            "rank": rank + 1,
            "score": float(cosine_sim),
            "nct_id": chunk["nct_id"],
            "title": chunk["title"],
            "text": chunk["text"],
        })
    return results


def show_results(query, results):
    """Pretty-print retrieval results for inspection."""
    print(f"\nQUERY: {query!r}\n")
    print("-" * 70)
    for r in results:
        print(f"Rank {r['rank']}  |  score={r['score']:.3f}  |  {r['nct_id']}")
        print(f"  Title: {r['title'][:80]}")
        snippet = r["text"][:200].replace("\n", " ")
        print(f"  Text:  {snippet}...")
        print()


# Test 1: the pembrolizumab/Keytruda case from Day 6
results = retrieve("Keytruda for non-small cell lung cancer", k=5)
show_results("Keytruda for non-small cell lung cancer", results)


QUERY: 'Keytruda for non-small cell lung cancer'

----------------------------------------------------------------------
Rank 1  |  score=0.930  |  NCT06532799
  Title: TIL Therapy Combined With Pembrolizumab for Advanced or Metastatic Refractory St
  Text:  This trial involves a multi-step treatment process. First, tumor samples are collected from patients for TIL extraction. Following this, a lymphodepletion regimen using cyclophosphamide and fludarabin...

Rank 2  |  score=0.929  |  NCT05280314
  Title: Phase II Trial of Neoadjuvant and Adjuvant IO102-IO103 and Pembrolizumab KEYTRUD
  Text:  Title: Phase II Trial of Neoadjuvant and Adjuvant IO102-IO103 and Pembrolizumab KEYTRUDA® in Patients With Resectable Tumors  Conditions: Melanoma, Squamous Cell Carcinoma of Head and Neck  Brief Summ...

Rank 3  |  score=0.926  |  NCT06532799
  Title: TIL Therapy Combined With Pembrolizumab for Advanced or Metastatic Refractory St
  Text:  Brief Summary: This Phase I/II study evaluates the saf

In [12]:
#Cell 10: Add deduplication to retrieval
def retrieve(query: str, k: int = 5, fetch_multiplier: int = 4):
    """
    Encode query, fetch (k * fetch_multiplier) chunks from FAISS, then
    keep only the best-scoring chunk per source trial. Return top-k unique trials.

    Why over-fetch + dedupe: multiple chunks from the same trial often score
    similarly. Without dedup, top-5 might be 3 trials. We over-fetch then keep
    the highest-scoring chunk per nct_id.
    """
    query_vec = model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)

    fetch_k = k * fetch_multiplier
    distances, indices = index.search(query_vec, fetch_k)

    # Walk results in score order; keep first hit per nct_id.
    seen_nct_ids = set()
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        chunk = chunks[idx]
        if chunk["nct_id"] in seen_nct_ids:
            continue
        seen_nct_ids.add(chunk["nct_id"])
        cosine_sim = 1 - (dist / 2)
        results.append({
            "rank": len(results) + 1,
            "score": float(cosine_sim),
            "nct_id": chunk["nct_id"],
            "title": chunk["title"],
            "text": chunk["text"],
        })
        if len(results) >= k:
            break
    return results


# Re-run the same query — should now return 5 unique trials
results = retrieve("Keytruda for non-small cell lung cancer", k=5)
show_results("Keytruda for non-small cell lung cancer", results)


QUERY: 'Keytruda for non-small cell lung cancer'

----------------------------------------------------------------------
Rank 1  |  score=0.930  |  NCT06532799
  Title: TIL Therapy Combined With Pembrolizumab for Advanced or Metastatic Refractory St
  Text:  This trial involves a multi-step treatment process. First, tumor samples are collected from patients for TIL extraction. Following this, a lymphodepletion regimen using cyclophosphamide and fludarabin...

Rank 2  |  score=0.929  |  NCT05280314
  Title: Phase II Trial of Neoadjuvant and Adjuvant IO102-IO103 and Pembrolizumab KEYTRUD
  Text:  Title: Phase II Trial of Neoadjuvant and Adjuvant IO102-IO103 and Pembrolizumab KEYTRUDA® in Patients With Resectable Tumors  Conditions: Melanoma, Squamous Cell Carcinoma of Head and Neck  Brief Summ...

Rank 3  |  score=0.919  |  NCT06694454
  Title: Neoadjuvant Inhaled Azacytidine With Platinum-Based Chemotherapy and Durvalumab 
  Text:  Brief Summary: Background:  Lung cancer is the leading

In [13]:
#Cell 11: Test battery + threshold calibration
TEST_QUERIES = [
    # In-domain: oncology, well-represented
    ("pembrolizumab non-small cell lung cancer", "in-domain"),
    ("hazard ratio overall survival immunotherapy", "in-domain"),
    ("HER2 positive breast cancer treatment", "in-domain"),
    ("BRAF mutation melanoma", "in-domain"),

    # In-domain: cardio/diabetes (less coverage but present)
    ("type 2 diabetes treatment", "in-domain"),
    ("heart failure ejection fraction", "in-domain"),

    # In-domain but specific (the sneaky case from Day 6)
    ("insulin dosage in pregnant women with type 1 diabetes", "in-domain-specific"),

    # Out-of-domain: should fail
    ("how do I bake sourdough bread", "out-of-domain"),
    ("python list comprehension syntax", "out-of-domain"),
    ("best hiking trails in New Hampshire", "out-of-domain"),
]

print(f"{'Category':<22} {'Top score':<12} {'Top NCT':<14} Query")
print("-" * 110)

for query, category in TEST_QUERIES:
    results = retrieve(query, k=3)
    top_score = results[0]["score"]
    top_nct = results[0]["nct_id"]
    print(f"{category:<22} {top_score:<12.3f} {top_nct:<14} {query[:60]}")

Category               Top score    Top NCT        Query
--------------------------------------------------------------------------------------------------------------
in-domain              0.941        NCT02991482    pembrolizumab non-small cell lung cancer
in-domain              0.919        NCT05675410    hazard ratio overall survival immunotherapy
in-domain              0.941        NCT04836156    HER2 positive breast cancer treatment
in-domain              0.923        NCT02224781    BRAF mutation melanoma
in-domain              0.928        NCT01106690    type 2 diabetes treatment
in-domain              0.928        NCT04307147    heart failure ejection fraction
in-domain-specific     0.939        NCT06280703    insulin dosage in pregnant women with type 1 diabetes
out-of-domain          0.857        NCT05058859    how do I bake sourdough bread
out-of-domain          0.864        NCT03138473    python list comprehension syntax
out-of-domain          0.844        NCT05632653    b

In [14]:
#Cell 12: Groq Llama with refusal clause
from groq import Groq

# Parameterize model name (Day 6 lesson: Groq deprecated a model on you mid-build)
LLM_MODEL = "llama-3.3-70b-versatile"
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

SYSTEM_PROMPT = """You are a clinical trial research assistant. You answer questions \
strictly using the trial excerpts provided in the context. Follow these rules:

1. If the context does not contain enough information to answer, say:
   "I don't have enough information in the retrieved trials to answer that question."
   Do not guess. Do not use general medical knowledge.

2. Cite every claim by NCT ID in square brackets, e.g. [NCT01234567]. If a claim
   is supported by multiple trials, cite all of them.

3. If the question is not about clinical trials or medical research, say:
   "This question is outside the scope of the clinical trial database."

4. Be concise. Two to four sentences unless the question requires more.
"""

def build_context(results):
    """Concatenate retrieved chunks with source markers for the LLM."""
    blocks = []
    for r in results:
        blocks.append(
            f"[{r['nct_id']}] {r['title']}\n{r['text']}"
        )
    return "\n\n---\n\n".join(blocks)


def generate(query: str, k: int = 5, threshold: float = 0.50, verbose: bool = True):
    """
    Full RAG pipeline: retrieve, threshold-check, generate (with refusal clause).
    """
    results = retrieve(query, k=k)
    top_score = results[0]["score"]

    # Layer 1 safety: similarity threshold (set low — catches only gibberish)
    if top_score < threshold:
        return {
            "answer": "I don't have enough information in the retrieved trials to answer that question.",
            "sources": [],
            "top_score": top_score,
            "refused_at": "threshold",
        }

    # Layer 2 safety: LLM refusal clause is in the SYSTEM_PROMPT
    context = build_context(results)
    user_message = f"Context:\n\n{context}\n\n---\n\nQuestion: {query}"

    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ],
        temperature=0.1,  # low temp = more deterministic, less hallucination
        max_tokens=400,
    )
    answer = response.choices[0].message.content

    if verbose:
        print(f"Top score: {top_score:.3f} (threshold={threshold})")
        print(f"Sources retrieved: {[r['nct_id'] for r in results]}")
        print(f"\nAnswer:\n{answer}\n")

    return {
        "answer": answer,
        "sources": [r["nct_id"] for r in results],
        "top_score": top_score,
        "refused_at": None,
    }


# Run one in-domain query to make sure the wire-up works
print("=" * 70)
print("TEST 1: In-domain medical query")
print("=" * 70)
_ = generate("What trials are testing pembrolizumab for non-small cell lung cancer?")

TEST 1: In-domain medical query
Top score: 0.947 (threshold=0.5)
Sources retrieved: ['NCT05709821', 'NCT06496009', 'NCT02991482', 'NCT02760225', 'NCT04252365']

Answer:
The trials testing pembrolizumab for non-small cell lung cancer are [NCT05709821], [NCT06496009], [NCT02760225], and [NCT04252365]. These trials are evaluating pembrolizumab as a treatment option for patients with non-small cell lung cancer, either alone or in combination with other therapies. [NCT05709821] is testing IMM60 with or without pembrolizumab, while [NCT04252365] is comparing pembrolizumab to sintilimab.



In [15]:
# Cell 13: Test the refusal clause
REFUSAL_TEST_QUERIES = [
    # Layer 2 must catch: scored 0.84-0.86 on Cell 11, well above 0.50 threshold
    ("how do I bake sourdough bread", "non-medical / Layer 2 should refuse"),
    ("python list comprehension syntax", "non-medical / Layer 2 should refuse"),

    # Sneaky case: scored 0.939, LLM has to recognize chunks don't answer it
    ("what is the recommended insulin dosage for pregnant women with type 1 diabetes",
     "in-domain-specific / Layer 2 should refuse if corpus lacks specifics"),

    # Genuinely answerable in-domain query as a control
    ("what trials are studying BRAF mutations in melanoma",
     "answerable / should produce cited answer"),
]

for query, expected_behavior in REFUSAL_TEST_QUERIES:
    print("=" * 70)
    print(f"QUERY: {query}")
    print(f"EXPECTED: {expected_behavior}")
    print("=" * 70)
    _ = generate(query)
    print()

QUERY: how do I bake sourdough bread
EXPECTED: non-medical / Layer 2 should refuse
Top score: 0.857 (threshold=0.5)
Sources retrieved: ['NCT05058859', 'NCT03278483', 'NCT06005142', 'NCT07513883', 'NCT06549556']

Answer:
This question is outside the scope of the clinical trial database.


QUERY: python list comprehension syntax
EXPECTED: non-medical / Layer 2 should refuse
Top score: 0.864 (threshold=0.5)
Sources retrieved: ['NCT03138473', 'NCT01174121', 'NCT03596541', 'NCT06672380', 'NCT07064174']

Answer:
This question is outside the scope of the clinical trial database.


QUERY: what is the recommended insulin dosage for pregnant women with type 1 diabetes
EXPECTED: in-domain-specific / Layer 2 should refuse if corpus lacks specifics
Top score: 0.934 (threshold=0.5)
Sources retrieved: ['NCT06280703', 'NCT02926937', 'NCT01709123', 'NCT00804986', 'NCT01098461']

Answer:
I don't have enough information in the retrieved trials to answer that question.


QUERY: what trials are studying BR